In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Homework6") \
    .master("local[*]") \
    .getOrCreate()

In [7]:
print(f"Spark version: {spark.version}")


Spark version: 4.0.2


In [8]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

# Check file size
!ls -lh yellow_tripdata_2025-11.parquet

--2026-03-10 07:13:05--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.73, 54.230.248.205, 54.230.248.64, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M   272MB/s    in 0.2s    

2026-03-10 07:13:06 (272 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]

-rw-r--r-- 1 root root 68M Dec 19 15:51 yellow_tripdata_2025-11.parquet


In [9]:
#Read the Yellow Taxi data
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# Check what we loaded
print(f"Total rows: {df.count():,}")
print("\nSchema:")
df.printSchema()

# Show sample data
df.show(5)

Total rows: 4,181,444

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+-------------

In [10]:
# Repartition the DataFrame to 4 partitions
df_repartitioned = df.repartition(4)

# Verify number of partitions
print(f"Number of partitions: {df_repartitioned.rdd.getNumPartitions()}")

Number of partitions: 4


In [11]:
# Define output path
output_path = "yellow_taxi_repartitioned"

# Save to parquet with 4 partitions
df_repartitioned.write.mode("overwrite").parquet(output_path)

print(f"✓ Data saved to {output_path}")

✓ Data saved to yellow_taxi_repartitioned


In [12]:
# List parquet files and their sizes
!ls -lh {output_path}/*.parquet

-rw-r--r-- 1 root root 26M Mar 10 07:13 yellow_taxi_repartitioned/part-00000-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar 10 07:13 yellow_taxi_repartitioned/part-00001-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar 10 07:13 yellow_taxi_repartitioned/part-00002-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar 10 07:13 yellow_taxi_repartitioned/part-00003-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet


In [13]:
import os
import glob

# Get all parquet files
parquet_files = glob.glob(f"{output_path}/*.parquet")

# Get sizes in bytes
sizes = [os.path.getsize(f) for f in parquet_files]

# Calculate average in MB
avg_size_mb = sum(sizes) / len(sizes) / (1024 * 1024)

print(f"\nNumber of .parquet files: {len(parquet_files)}")
print(f"File sizes (MB):")
for f, size in zip(parquet_files, sizes):
    print(f"  {os.path.basename(f)}: {size / (1024 * 1024):.2f} MB")

print(f"\nAverage size: {avg_size_mb:.2f} MB")


Number of .parquet files: 4
File sizes (MB):
  part-00000-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet: 25.35 MB
  part-00001-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet: 25.33 MB
  part-00002-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet: 25.33 MB
  part-00003-b2cf177a-471a-41a4-b484-a5338e288103-c000.snappy.parquet: 25.34 MB

Average size: 25.34 MB


In [14]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [15]:
df.printSchema()

# Show sample data
df.select("tpep_pickup_datetime", "tpep_dropoff_datetime").show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------------------+---------------------+
|tpep_pickup_datetime|tpep_dro

In [16]:
from pyspark.sql import functions as F

In [17]:
# Filter trips that started on November 15, 2025
trips_nov_15 = df.filter(
    F.to_date(F.col("tpep_pickup_datetime")) == "2025-11-15"
)

# Show sample to verify
print("Sample trips on Nov 15:")
trips_nov_15.select("tpep_pickup_datetime").show(5)

Sample trips on Nov 15:
+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2025-11-15 00:00:16|
| 2025-11-15 00:03:01|
| 2025-11-15 00:48:45|
| 2025-11-15 00:28:25|
| 2025-11-15 00:00:00|
+--------------------+
only showing top 5 rows


In [18]:
# Count trips on November 15
count = trips_nov_15.count()

print(f"\nTotal trips on November 15, 2025: {count:,}")


Total trips on November 15, 2025: 162,604


In [19]:
# See the pickup and dropoff columns
df.select("tpep_pickup_datetime", "tpep_dropoff_datetime").show(5)

+--------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------------------+---------------------+
| 2025-11-01 00:13:25|  2025-11-01 00:13:25|
| 2025-11-01 00:49:07|  2025-11-01 01:01:22|
| 2025-11-01 00:07:19|  2025-11-01 00:20:41|
| 2025-11-01 00:00:00|  2025-11-01 01:01:03|
| 2025-11-01 00:18:50|  2025-11-01 00:49:32|
+--------------------+---------------------+
only showing top 5 rows


In [20]:
# Calculate duration in hours
df_with_duration = df.withColumn(
    "duration_hours",
    (F.unix_timestamp("tpep_dropoff_datetime") -
     F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

# Show sample durations
print("Sample trip durations:")
df_with_duration.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "duration_hours"
).show(5)

Sample trip durations:
+--------------------+---------------------+-------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|     duration_hours|
+--------------------+---------------------+-------------------+
| 2025-11-01 00:13:25|  2025-11-01 00:13:25|                0.0|
| 2025-11-01 00:49:07|  2025-11-01 01:01:22|0.20416666666666666|
| 2025-11-01 00:07:19|  2025-11-01 00:20:41|0.22277777777777777|
| 2025-11-01 00:00:00|  2025-11-01 01:01:03|             1.0175|
| 2025-11-01 00:18:50|  2025-11-01 00:49:32| 0.5116666666666667|
+--------------------+---------------------+-------------------+
only showing top 5 rows


In [21]:
# Find the longest trip
max_duration = df_with_duration.agg(
    F.max("duration_hours").alias("max_duration")
).collect()[0]["max_duration"]

print(f"\nLongest trip duration: {max_duration:.1f} hours")


Longest trip duration: 90.6 hours


In [23]:
# Download the zone lookup CSV
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

# Check the file
!head -10 taxi_zone_lookup.csv

--2026-03-10 07:19:57--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.205, 54.230.248.73, 54.230.248.222, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 07:19:58 (254 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","B

In [24]:
# Read the CSV file
zones_df = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

# Check the schema
zones_df.printSchema()

# Show sample data
print("Zone lookup data:")
zones_df.show(10)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Zone lookup data:
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+---------

In [25]:
# See what columns we have
print("Taxi data columns:")
print(df.columns)

print("\nZone lookup columns:")
print(zones_df.columns)

Taxi data columns:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

Zone lookup columns:
['LocationID', 'Borough', 'Zone', 'service_zone']


In [26]:
# Join on LocationID = PULocationID
trips_with_zones = df.join(
    zones_df,
    df.PULocationID == zones_df.LocationID,
    "left"  # Left join to keep all trips
)

# Verify the join
print("Sample trips with zone names:")
trips_with_zones.select(
    "PULocationID",
    "LocationID",
    "Zone",
    "Borough"
).show(10)

Sample trips with zone names:
+------------+----------+--------------------+---------+
|PULocationID|LocationID|                Zone|  Borough|
+------------+----------+--------------------+---------+
|          43|        43|        Central Park|Manhattan|
|         142|       142| Lincoln Square East|Manhattan|
|         163|       163|       Midtown North|Manhattan|
|         138|       138|   LaGuardia Airport|   Queens|
|         138|       138|   LaGuardia Airport|   Queens|
|          90|        90|            Flatiron|Manhattan|
|         142|       142| Lincoln Square East|Manhattan|
|         237|       237|Upper East Side S...|Manhattan|
|         162|       162|        Midtown East|Manhattan|
|         234|       234|            Union Sq|Manhattan|
+------------+----------+--------------------+---------+
only showing top 10 rows


In [27]:
# Count pickups by zone
pickup_counts = trips_with_zones.groupBy("Zone") \
    .count() \
    .orderBy("count")  # Order by count ascending

# Show all zones with counts
print("Pickup counts by zone (showing all):")
pickup_counts.show(300, truncate=False)

Pickup counts by zone (showing all):
+---------------------------------------------+------+
|Zone                                         |count |
+---------------------------------------------+------+
|Governor's Island/Ellis Island/Liberty Island|1     |
|Eltingville/Annadale/Prince's Bay            |1     |
|Arden Heights                                |1     |
|Port Richmond                                |3     |
|Rikers Island                                |4     |
|Rossville/Woodrow                            |4     |
|Great Kills                                  |4     |
|Green-Wood Cemetery                          |4     |
|Jamaica Bay                                  |5     |
|Westerleigh                                  |12    |
|New Dorp/Midland Beach                       |14    |
|Oakwood                                      |14    |
|Crotona Park                                 |14    |
|West Brighton                                |14    |
|Willets Point              

In [28]:
# Get the zone with minimum pickups
least_frequent = pickup_counts.first()

print(f"\nLeast frequent pickup zone:")
print(f"  Zone: {least_frequent['Zone']}")
print(f"  Pickups: {least_frequent['count']}")


Least frequent pickup zone:
  Zone: Governor's Island/Ellis Island/Liberty Island
  Pickups: 1
